### Current (11/05/2025) training pipeline

In [11]:
import datasets as hf_datasets

from workspace.sources.local_datasets.dataset import Dataset
from workspace.sources.local_datasets.preprocessing.transformations import Truncation, HuggingFaceDatasetConversion
from workspace.sources.local_datasets.preprocessing.tokenization import BertBaseUncasedTokenizer
from workspace.sources.local_datasets.preprocessing.encoding import BertBaseUncasedEncoder
from workspace.sources.local_datasets.data_classes import HuggingFaceData
from workspace.sources.experiments.metrics import Loss
from workspace.sources.models.bert_base_uncased import BertBaseUncased


import mlflow

mlflow.set_tracking_uri('../../../mlruns')
mlflow.set_experiment('test_old_vs_new_training')

<Experiment: artifact_location='file:C:/Users/nikita.bardatskii/git/bi-bap-2025-bardanik/workspace/tests/../mlruns/328346920420097447', creation_time=1746961887234, experiment_id='328346920420097447', last_update_time=1746961887234, lifecycle_stage='active', name='test_old_vs_new_training', tags={}>

In [12]:
random_state = 2703938567

In [13]:
recovery = Dataset('ReCovery',
                   '../../../sources/local_datasets/data/Recovery/recovery.csv').init(random_state=random_state)
train_set_data = recovery.train_set
val_set_data = recovery.val_set
test_set_data = recovery.test_set
train_set_data.dataset.head()


2025-05-11 17:20:38,209 - Dataset - INFO - mlflow is not active, could not log the params: {'preprocessing_pipeline_name': 'empty', 'preprocessing_pipeline_representation': "<PreprocessingPipeline 'empty': []>", 'preprocessing_pipeline': []}
2025-05-11 17:20:38,209 - Dataset - INFO - Prepared dataset does not exist, computing from scratch.
2025-05-11 17:20:38,348 - Dataset - INFO - Initializing preprocessing pipeline: <PreprocessingPipeline 'empty': []>
2025-05-11 17:20:38,348 - Dataset - INFO - artifacts path is none, skipping saving.


,article,label
1117,Study Finds That Cloth Masks Can Increase Heal...,0
1080,How to have an election during COVID-19: India...,1
1015,Family of Amazon employee who died of COVID-19...,1
915,Calls To Helplines In America Increase Dramati...,0
622,Tommy Bartlett Show cancels 2020 season due to...,1


In [14]:
truncated_train_set_data = Truncation().init().preprocess(train_set_data.copy())
truncated_val_set_data = Truncation().init().preprocess(val_set_data.copy())
truncated_test_set_data = Truncation().init().preprocess(test_set_data.copy())
truncated_train_set_data.dataset

,article,label
1117,Study Finds That Cloth Masks Can Increase Heal...,0
1080,How to have an election during COVID-19: India...,1
1015,Family of Amazon employee who died of COVID-19...,1
915,Calls To Helplines In America Increase Dramati...,0
622,Tommy Bartlett Show cancels 2020 season due to...,1
...,...,...
74,Trump Steps In To Help Oil Industry Facing Its...,1
1688,At least a quarter of the workforce is out of ...,1
1805,Will virus keep Florida spectators from astron...,1
681,WHO Guidelines for Frontline PPE Use Designed ...,1


In [15]:
tokenized_train_set_data = BertBaseUncasedTokenizer().init().preprocess(truncated_train_set_data.copy())
tokenized_train_set_data.dataset

,article,label
1117,"[study, finds, that, cloth, masks, can, increa...",0
1080,"[how, to, have, an, election, during, co, ##vi...",1
1015,"[family, of, amazon, employee, who, died, of, ...",1
915,"[calls, to, help, ##lines, in, america, increa...",0
622,"[tommy, bartlett, show, cancel, ##s, 2020, sea...",1
...,...,...
74,"[trump, steps, in, to, help, oil, industry, fa...",1
1688,"[at, least, a, quarter, of, the, workforce, is...",1
1805,"[will, virus, keep, florida, spectators, from,...",1
681,"[who, guidelines, for, front, ##line, pp, ##e,...",1


In [7]:
encoded_train_set_data = BertBaseUncasedEncoder().init().preprocess(
    tokenized_train_set_data.copy()
)
encoded_train_set_data.features

,input_ids,token_type_ids,attention_mask,article
1117,"[101, 2025, 4090, 1115, 8217, 17944, 1169, 277...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",NaN
1080,"[101, 1293, 1106, 1138, 1126, 1728, 1219, 1884...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",NaN
1015,"[101, 1266, 1104, 1821, 10961, 1320, 7775, 115...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",NaN
915,"[101, 3675, 1106, 1494, 108, 108, 2442, 1107, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",NaN
622,"[101, 1106, 16211, 2927, 5034, 3069, 1437, 197...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",NaN
...,...,...,...,...
74,NaN,NaN,NaN,"[trump, steps, in, to, help, oil, industry, fa..."
1688,NaN,NaN,NaN,"[at, least, a, quarter, of, the, workforce, is..."
1805,NaN,NaN,NaN,"[will, virus, keep, florida, spectators, from,..."
681,NaN,NaN,NaN,"[who, guidelines, for, front, ##line, pp, ##e,..."


In [9]:
converted_train_set_data = HuggingFaceDatasetConversion().init().preprocess(encoded_train_set_data.copy())
converted_train_set_data.dataset

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'label', '__index_level_0__'],
    num_rows: 142
})

In [11]:
type(converted_train_set_data.dataset)

datasets.arrow_dataset.Dataset

### Old training pipeline

In [7]:
tokenizer_prepr = BertBaseUncasedTokenizer().init()
tokenizer = tokenizer_prepr.tokenizer

In [8]:
hf_train_set = hf_datasets.Dataset.from_pandas(truncated_train_set_data.dataset)
hf_val_set = hf_datasets.Dataset.from_pandas(truncated_val_set_data.dataset)
hf_test_set = hf_datasets.Dataset.from_pandas(truncated_test_set_data.dataset)
hf_train_set

Dataset({
    features: ['article', 'label', '__index_level_0__'],
    num_rows: 142
})

In [9]:
class Encoder:
    print_once = False
    @staticmethod
    def encoding_fn(example):
        output = tokenizer(example['article'],
                           padding='max_length',
                           truncation=True,
                           max_length=512)
        if Encoder.print_once:
            print(example)
            print(example['article'])
            print(output)
            Encoder.print_once = False
        return output

hf_train_set_encoded = hf_train_set.map(Encoder.encoding_fn, batched=True)
hf_val_set_encoded = hf_val_set.map(Encoder.encoding_fn, batched=True)
hf_test_set_encoded = hf_test_set.map(Encoder.encoding_fn, batched=True)
hf_train_set_encoded.set_format(type='torch')
hf_val_set_encoded.set_format(type='torch')
hf_test_set_encoded.set_format(type='torch')

Map:   0%|          | 0/142 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

In [65]:
converted_train_set_data.dataset[0]

{'input_ids': tensor([  101,  2025,  4090,  1115,  8217, 17944,  1169,  2773, 12520,  3239,
          3187,  1104,  8974,  1107,  4094,  1103,  9193,   131,   170,  2025,
          1502,  1107,  1410,  1276,  1115,  8217, 17944,  1169,  2773, 12520,
          3239,  3187,  1104,  8974,   119,  1122,  1145,  1270,  1154,  2304,
          1103, 23891,  1104,  2657, 17944,   119,  7977,  1113,   131,  1191,
         17944,  1336,  1136,  3244, 12520,  3239,  1107,  1126, 12104,  3545,
           117,  1184,  1132,  1152,  1833,  1111,  1103,  1470,   136,  1132,
          1103,  6134,  1189,  1118,  2332, 12638,  6421,  1579,  1107,  1103,
          1436,  2199,  1104,  1103,  1470,   136,   170,  1974,  1104,  2844,
          1132,  1299,   108,   108,  4676,  1115,  1234,  4330,   170,  7739,
           119,  1199, 14206,  4822,  1303,  1107,  1169,  7971,  1132,  1543,
          1122, 11839,  1111,  1234,  1150,  1328,  1106,  1202,  1199,  6001,
           117,  1105, 12724,  8805,  1

In [66]:
hf_train_set_encoded[0]

{'article': 'Study Finds That Cloth Masks Can Increase Healthcare Workers Risk of Infection In Brief The Facts: A study published in 2015 found that cloth masks can increase healthcare workers risk of infection. It also called into question the efficacy of medical masks.\r\n\r\nReflect On: If masks may not protect healthcare workers in an acute setting, what are they doing for the public? Are the decisions made by health regulatory agencies always in the best interest of the public?\r\n\r\nA lot of places are mandating that people wear a mask. Some grocery stores here in Canada are making it mandatory for people who want to do some shopping, and Los Angeles County recently mandated that all people must wear a mask when going outside. But do these measures really help? We are living in strange times when people like Bill Gates are getting a lot of T.V. time, as he seems to be the world’s leading ‘health’ authority on the new coronavirus, what we should do, and how we’re going to stop it

In [61]:
print(type(converted_train_set_data.dataset['input_ids']))
converted_train_set_data.dataset['input_ids'] == hf_train_set_encoded['input_ids']

<class 'torch.Tensor'>


tensor([[ True, False, False,  ..., False, False,  True],
        [ True, False, False,  ..., False, False,  True],
        [ True, False, False,  ..., False, False,  True],
        ...,
        [ True, False, False,  ..., False, False,  True],
        [ True, False, False,  ...,  True,  True,  True],
        [ True, False, False,  ..., False, False,  True]])

In [63]:
converted_train_set_data.dataset['input_ids']

tensor([[  101,  2025,  4090,  ...,  2369,   113,   102],
        [  101,  1293,  1106,  ...,   108,   182,   102],
        [  101,  1266,  1104,  ...,  1412,  3578,   102],
        ...,
        [  101,  1209,  7942,  ...,  1306,  1115,   102],
        [  101,  1150, 13112,  ...,     0,     0,     0],
        [  101,   786,  5372,  ...,  1198,  1660,   102]])

In [62]:
hf_train_set_encoded['input_ids']

tensor([[  101,  2817,  4858,  ...,  2030,  4381,   102],
        [  101,  2129,  2000,  ...,  6224,  9930,   102],
        [  101,  2155,  1997,  ...,  2129,  2116,   102],
        ...,
        [  101,  2097,  7865,  ...,  2015,  1012,   102],
        [  101,  2040, 11594,  ...,     0,     0,     0],
        [  101,  1520,  2882,  ...,  3901,  1010,   102]])

In [10]:

with mlflow.start_run() as run:
    recovery.init(random_state=random_state)
    recovery.preprocessed_train_set = HuggingFaceData(hf_train_set_encoded)
    recovery.preprocessed_val_set = HuggingFaceData(hf_val_set_encoded)
    recovery.preprocessed_test_set = HuggingFaceData(hf_test_set_encoded)
    recovery.save_prepared_datasets()
    model_training_arguments = {'epochs': 3,
                                'early_stopping_patience': None,
                                'early_stopping_threshold': None}

    model = BertBaseUncased(train_best_model_metric=Loss,
                            training_arguments=model_training_arguments)
    model.init(random_state=random_state)
    model.fit(recovery)
    model.evaluate()

2025-05-11 16:49:12,593 - Dataset - INFO - Prepared dataset does not exist, computing from scratch.
2025-05-11 16:49:12,742 - Dataset - INFO - Initializing preprocessing pipeline: <PreprocessingPipeline 'empty': []>


Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,0.559900,No log


KeyError: "The `metric_for_best_model` training argument is set to 'eval_loss', which is not found in the evaluation metrics. The available evaluation metrics are: []. Consider changing the `metric_for_best_model` via the TrainingArguments."

### Mapping already encoded

In [48]:
class EncoderFromTokenized:
    print_once = False
    @staticmethod
    def encoding_fn(example):
        output = tokenizer(example['article'],
                           is_split_into_words=True,
                           padding='max_length',
                           truncation=True,
                           max_length=512)
        if EncoderFromTokenized.print_once:
            print(example)
            print(example['article'])
            print(output)
            EncoderFromTokenized.print_once = False
        return output

In [16]:
hf_tokenized_train_set = hf_datasets.Dataset.from_pandas(tokenized_train_set_data.dataset)
hf_tokenized_train_set

Dataset({
    features: ['article', 'label', '__index_level_0__'],
    num_rows: 142
})

In [19]:
hf_tokenized_train_set.set_format(type='torch')
print(hf_tokenized_train_set.format)
hf_tokenized_train_set

{'type': 'torch', 'format_kwargs': {}, 'columns': ['article', 'label', '__index_level_0__'], 'output_all_columns': False}


Dataset({
    features: ['article', 'label', '__index_level_0__'],
    num_rows: 142
})

In [23]:
hf_tokenized_train_set_encoded = hf_tokenized_train_set.map(EncoderFromTokenized.encoding_fn, batched=True)


Map:   0%|          | 0/142 [00:00<?, ? examples/s]

In [26]:
hf_tokenized_train_set_encoded['input_ids']

tensor([[  101,  2817,  4858,  ...,  2591,  2865,   102],
        [  101,  2129,  2000,  ...,  2000,  9689,   102],
        [  101,  2155,  1997,  ...,  1055,  2331,   102],
        ...,
        [  101,  2097,  7865,  ...,  8911,  3958,   102],
        [  101,  2040, 11594,  ...,     0,     0,     0],
        [  101,  1520,  2882,  ...,  4394,  2041,   102]])

In [27]:
hf_train_set_encoded['input_ids']

tensor([[  101,  2817,  4858,  ...,  2030,  4381,   102],
        [  101,  2129,  2000,  ...,  6224,  9930,   102],
        [  101,  2155,  1997,  ...,  2129,  2116,   102],
        ...,
        [  101,  2097,  7865,  ...,  2015,  1012,   102],
        [  101,  2040, 11594,  ...,     0,     0,     0],
        [  101,  1520,  2882,  ...,  3901,  1010,   102]])

In [28]:
hf_tokenized_train_set_encoded['input_ids'] == hf_train_set_encoded['input_ids']

tensor([[ True,  True,  True,  ..., False, False,  True],
        [ True,  True,  True,  ..., False, False,  True],
        [ True,  True,  True,  ..., False, False,  True],
        ...,
        [ True,  True,  True,  ..., False, False,  True],
        [ True,  True,  True,  ...,  True,  True,  True],
        [ True,  True,  True,  ..., False, False,  True]])

In [36]:
truncated_train_set_data.features.iloc[5]

'Lyft is offering free e-scooter trips to health care workers fighting COVID-19 Lyft will provide free electric scooter rides to essential workers in response to the COVID-19 pandemic. While most scooter-sharing programs in the US have temporarily ceased operations in response to the pandemic, Lyft is still operating scooters in a handful of cities, and the company wants to support the transportation needs of health care workers and others on the frontlines of the crisis.\r\n\r\nLyft is offering free scooter rides through April 30th, up to 30 minutes in length, for first-responders, health care employees, and transit workers in the following cities: Austin, Denver, Los Angeles, San Diego, Santa Monica, and Washington, DC. Employers like hospitals, clinics, and transit agencies can email Lyft at heroscooters@lyft.com to obtain enrollment information that they can distribute to staff.\r\n\r\nLyft says it will prioritize scooter deployment near local hospitals for health care workers. The

In [45]:
tokenizer.convert_ids_to_tokens(hf_tokenized_train_set_encoded['input_ids'][5][:6])

['[CLS]', 'l', '#', '#', 'y', '#']

In [46]:
tokenizer.convert_ids_to_tokens(hf_train_set_encoded['input_ids'][5][:6])

['[CLS]', 'l', '##y', '##ft', 'is', 'offering']

In [50]:
enc = tokenizer("Lyft is offering")
tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"])
print(tokens)

['[CLS]', 'l', '##y', '##ft', 'is', 'offering', '[SEP]']


In [54]:
words = tokenizer.tokenize("Lyft is offering")            # → ['un',
enc2 = tokenizer(words, is_split_into_words=True)
tokens2 = tokenizer.convert_ids_to_tokens(enc2["input_ids"])
print(tokens2)

['[CLS]', 'l', '#', '#', 'y', '#', '#', 'ft', 'is', 'offering', '[SEP]']


In [55]:
words = ['Lyft', 'is', 'offering']
enc2 = tokenizer(words, is_split_into_words=True)
tokens2 = tokenizer.convert_ids_to_tokens(enc2["input_ids"])
print(tokens2)

['[CLS]', 'l', '##y', '##ft', 'is', 'offering', '[SEP]']
